# 00 · Setup (Colab A100)

Chạy notebook này **một lần mỗi session** trước khi chạy model.
Dữ liệu RIÊNG TƯ không nằm trong repo — bạn upload lên Google Drive rồi mount vào.

**Chuẩn bị trên Drive** (thư mục `MyDrive/asr_benchmark_data/`):
- `Auto_AutoELE/` (wav), `SOL_UPS/` (wav)
- `Data Audio_Auto_AutoELE_Grid.csv`, `Data Audio_SOL_UPS_Grid.csv`


In [ ]:
!nvidia-smi -L

In [ ]:
# Clone repo
![ -d benchmark-for-asr-model ] || git clone https://github.com/jajqja/benchmark-for-asr-model.git
%cd benchmark-for-asr-model
!git pull -q

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE_DATA = '/content/drive/MyDrive/asr_benchmark_data'  # <-- sửa nếu bạn để nơi khác
assert os.path.isdir(DRIVE_DATA), f'Không thấy {DRIVE_DATA} — upload data lên Drive trước'

# Link data từ Drive vào repo để config (đường dẫn tương đối) dùng được nguyên trạng
for name in ['Auto_AutoELE', 'SOL_UPS',
             'Data Audio_Auto_AutoELE_Grid.csv', 'Data Audio_SOL_UPS_Grid.csv']:
    src = os.path.join(DRIVE_DATA, name)
    if os.path.exists(src) and not os.path.exists(name):
        os.symlink(src, name)
    print(('OK  ' if os.path.exists(name) else 'MISS') + '  ' + name)

In [ ]:
# Kết quả (hypotheses/metrics) lưu ra Drive -> resume được khi session chết
RESULTS_DRIVE = '/content/drive/MyDrive/asr_benchmark_results'
os.makedirs(RESULTS_DRIVE, exist_ok=True)
!rm -rf results && ln -s $RESULTS_DRIVE results
!mkdir -p results/hypotheses results/metrics results/reports
print('results ->', os.path.realpath('results'))

In [ ]:
# Deps lõi + build manifest cho cả 2 bộ
!pip install -q -r requirements/base.txt
!python scripts/prepare_manifest.py --config configs/datasets/auto_autoele.yaml
!python scripts/prepare_manifest.py --config configs/datasets/sol_ups.yaml
!wc -l data/manifests/*.jsonl